# PrimusM-MAE: nine downstream segmentation datasets, one epoch each

This cluster notebook reproduces the local compatibility test against every uploaded OpenMind-preprocessed
downstream dataset. Each task loads the released PrimusM MAE checkpoint, runs 250 training and 50 validation
patch iterations, records loss/pseudo Dice/memory/runtime, and continues after per-dataset failures.

It creates cluster-specific plan copies and does not edit the uploaded `PMPrep.json` files or `.b2nd` volumes.

In [ ]:
from __future__ import annotations

import gc
import json
import os
import sys
import time
import traceback
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

NNSSL_REPO = Path(os.environ.get("NNSSL_REPO", "/home/u6mn/taotl.u6mn/u6mn/nnssl"))
NNUNET_REPO = Path(os.environ.get("NNUNET_ADAPTATION_REPO", "/home/u6mn/taotl.u6mn/u6mn/nnUNet-openmind"))
PREPROCESSED_ROOT = Path(os.environ.get(
    "OPENMIND_DOWNSTREAM_PREPROCESSED",
    "/lus/lfs1aip2/projects/u6mn/datasets/OpenMind_downstream/segmentation_preprocessed",
))
RESULTS_ROOT = Path(os.environ.get(
    "OPENMIND_DOWNSTREAM_RESULTS",
    "/lus/lfs1aip2/projects/u6mn/openmind_jepa/downstream_primus_smoke",
))
CHECKPOINT = Path(os.environ.get(
    "PRIMUS_MAE_CHECKPOINT",
    NNSSL_REPO / "weights/PrimusM-OpenMind-MAE/checkpoint_final.pth",
))
TRAINER_NAME = os.environ.get("NNUNET_PRIMUS_TRAINER", "PMMAE1")
SOURCE_PLAN = "PMPrep"
CLUSTER_PLAN = "PMPrepCluster"
BATCH_SIZE = 2
DATASET_IDS = list(range(201, 210))

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
os.environ.update(
    nnUNet_preprocessed=str(PREPROCESSED_ROOT), nnUNet_results=str(RESULTS_ROOT),
    nnUNet_compile="false", OMP_NUM_THREADS="8", MKL_NUM_THREADS="8",
)
if NNUNET_REPO.is_dir() and str(NNUNET_REPO) not in sys.path:
    sys.path.insert(0, str(NNUNET_REPO))
assert PREPROCESSED_ROOT.is_dir(), PREPROCESSED_ROOT
assert CHECKPOINT.is_file(), CHECKPOINT
print({"data": str(PREPROCESSED_ROOT), "results": str(RESULTS_ROOT), "checkpoint": str(CHECKPOINT)})

## 1. Discover and audit all uploaded datasets

In [ ]:
dataset_dirs = {int(path.name[7:10]): path for path in PREPROCESSED_ROOT.glob("Dataset???_*")}
missing = [dataset_id for dataset_id in DATASET_IDS if dataset_id not in dataset_dirs]
assert not missing, f"Datasets not uploaded yet: {missing}"

inventory = []
for dataset_id in DATASET_IDS:
    folder = dataset_dirs[dataset_id]
    dataset_json = json.loads((folder / "dataset.json").read_text())
    splits = json.loads((folder / "splits_final.json").read_text())
    data_folders = [path for path in folder.iterdir() if path.is_dir() and path.name.endswith("3d_fullres")]
    assert len(data_folders) == 1, (folder, data_folders)
    b2nd = list(data_folders[0].glob("*.b2nd"))
    inventory.append({
        "id": dataset_id, "dataset": folder.name, "channels": len(dataset_json["channel_names"]),
        "labels": len(dataset_json["labels"]), "train": len(splits[0]["train"]),
        "validation": len(splits[0]["val"]), "b2nd_files": len(b2nd),
    })
inventory_table = pd.DataFrame(inventory)
display(inventory_table)

## 2. Install runtime compatibility hooks

Released OpenMind property files store foreground coordinates as `(z,y,x)`, while some nnU-Net branches expect
`(class,z,y,x)`. The first hook adds a temporary dummy coordinate column before bounding-box selection. The second
maps sparse source label IDs (TopCoW uses `0..12,15`) to contiguous training channels. These hooks live only in the
current Python process and do not rewrite uploaded data.

In [ ]:
from nnunetv2.training.dataloading.data_loader import nnUNetDataLoader

if not getattr(nnUNetDataLoader, "_openmind_compat_installed", False):
    original_get_bbox = nnUNetDataLoader.get_bbox
    original_generate = nnUNetDataLoader.generate_train_batch

    def openmind_get_bbox(self, data_shape, force_fg, class_locations, *args, **kwargs):
        if class_locations is not None:
            normalized = {}
            spatial_dims = len(data_shape)
            for key, locations in class_locations.items():
                array = np.asarray(locations)
                if array.ndim == 2 and array.shape[1] == spatial_dims:
                    array = np.concatenate([np.zeros((len(array), 1), dtype=array.dtype), array], axis=1)
                normalized[key] = array
            class_locations = normalized
        return original_get_bbox(self, data_shape, force_fg, class_locations, *args, **kwargs)

    def remap_target(target, mapping):
        if isinstance(target, list):
            return [remap_target(item, mapping) for item in target]
        source = target.clone() if torch.is_tensor(target) else target.copy()
        for label_value, train_id in mapping.items():
            target[source == label_value] = train_id
        return target

    def openmind_generate(self):
        batch = original_generate(self)
        labels = list(self.annotated_classes_key[1:])
        mapping = {int(label): train_id for train_id, label in enumerate(labels) if int(label) != train_id}
        if mapping:
            batch["target"] = remap_target(batch["target"], mapping)
        return batch

    nnUNetDataLoader.get_bbox = openmind_get_bbox
    nnUNetDataLoader.generate_train_batch = openmind_generate
    nnUNetDataLoader._openmind_compat_installed = True
print("OpenMind coordinate and sparse-label compatibility hooks installed")

## 3. Create cluster plan copies pointing to the cluster checkpoint

In [ ]:
for dataset_id, folder in dataset_dirs.items():
    if dataset_id not in DATASET_IDS:
        continue
    source = folder / f"{SOURCE_PLAN}.json"
    assert source.is_file(), source
    plan = json.loads(source.read_text())
    plan["plans_name"] = CLUSTER_PLAN
    plan.setdefault("pretrain_info", {})["checkpoint_path"] = str(CHECKPOINT)
    target = folder / f"{CLUSTER_PLAN}.json"
    target.write_text(json.dumps(plan, indent=2), encoding="utf-8")
    print(target)

## 4. Run one epoch per dataset

`RUN_TRAINING` defaults to `False` so the path audit can be executed safely on a login/Jupyter node. Change it to
`True` inside a GPU allocation. Checkpoint writing is disabled because this is a numerical compatibility test.

In [ ]:
from nnunetv2.run.run_training import get_trainer_from_args

RUN_TRAINING = False
if RUN_TRAINING:
    assert torch.cuda.is_available(), "Run this cell in a GPU allocation"
    torch.manual_seed(12345)
    print("GPU:", torch.cuda.get_device_name())
    summaries = []
    for dataset_id in DATASET_IDS:
        started = time.perf_counter()
        summary = {"id": dataset_id, "dataset": dataset_dirs[dataset_id].name, "status": "failed"}
        trainer = None
        try:
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
            trainer = get_trainer_from_args(
                str(dataset_id), "3d_fullres", 0, TRAINER_NAME, CLUSTER_PLAN,
                device=torch.device("cuda"), pretrained_from_scratch=False,
                overwrite_ckpt_path=None,
            )
            trainer.num_epochs = 1
            trainer.num_iterations_per_epoch = 250
            trainer.num_val_iterations_per_epoch = 50
            trainer.batch_size = BATCH_SIZE
            trainer.configuration_manager.configuration["batch_size"] = BATCH_SIZE
            trainer.disable_checkpointing = True
            trainer.run_training()
            logs = trainer.logger.my_fantastic_logging
            summary.update(
                status="completed", train_loss=float(logs["train_losses"][-1]),
                val_loss=float(logs["val_losses"][-1]),
                pseudo_dice=float(logs.get("mean_fg_dice", [np.nan])[-1]),
                peak_allocated_gib=torch.cuda.max_memory_allocated() / 1024**3,
            )
        except Exception as error:
            summary.update(error_type=type(error).__name__, error=str(error), traceback=traceback.format_exc())
        finally:
            summary["elapsed_seconds"] = time.perf_counter() - started
            summaries.append(summary)
            (RESULTS_ROOT / "one_epoch_summary.json").write_text(json.dumps(summaries, indent=2))
            print(json.dumps(summary, indent=2))
            del trainer
            gc.collect()
            torch.cuda.empty_cache()
    results_table = pd.DataFrame(summaries)
    display(results_table)
else:
    print("Audit complete. Set RUN_TRAINING=True in this cell on a GPU node.")

## 5. Load existing results and visualize loss / pseudo Dice

In [ ]:
summary_path = RESULTS_ROOT / "one_epoch_summary.json"
assert summary_path.is_file(), f"Run the training cell first: {summary_path}"
results_table = pd.DataFrame(json.loads(summary_path.read_text()))
display(results_table)
completed = results_table[results_table.status == "completed"].copy()
assert len(completed) == len(DATASET_IDS), results_table[["dataset", "status", "error"]]
assert np.isfinite(completed[["train_loss", "val_loss"]].to_numpy()).all()

figure, axes = plt.subplots(1, 2, figsize=(15, 5))
x = np.arange(len(completed))
axes[0].bar(x - 0.2, completed.train_loss, width=0.4, label="train")
axes[0].bar(x + 0.2, completed.val_loss, width=0.4, label="validation")
axes[0].set_xticks(x, completed.dataset, rotation=65, ha="right")
axes[0].set_ylabel("loss")
axes[0].legend()
axes[1].bar(x, completed.pseudo_dice)
axes[1].set_xticks(x, completed.dataset, rotation=65, ha="right")
axes[1].set_ylabel("patch-level pseudo Dice")
for axis in axes:
    axis.grid(axis="y", alpha=0.25)
figure.tight_layout()
output = RESULTS_ROOT / "one_epoch_summary.png"
figure.savefig(output, dpi=180, bbox_inches="tight")
plt.show()
print(output)